### Update projected starting lineups

In [ ]:
# from MODELS.scrapStarting import NBADailyLineups

# scraper = NBADailyLineups("https://www.rotowire.com/basketball/nba-lineups.php")
# scraper.getDict()  # Scrape the lineups
# scraper.updateTeamInfo()  # Update teamInfo.py

Successfully updated /Users/alexgonzalez/Documents/NBA-Prop-Predictor/PRODUCTION/teamInfo.py
Updated 16 teams with confirmed lineups


In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
import sys
import os

project_root = os.path.dirname(os.path.abspath(''))
if project_root not in sys.path:
    sys.path.append(project_root)

from FEATURES.features import *
from FEATURES.featuresV2 import *
from PRODUCTION.calculateEVS import *
from PRODUCTION.pipeline import *
from PRODUCTION.teamInfo import teamStarPlayer, projectedStartingFive, mainStartingFive

### Load Model

In [2]:
# Load split NGBoost models (mean, variance, calibration factor, and isotonic calibrator)
pts_mean_model = joblib.load('../MODELS/SAVED_MODELS/NGBOOST_PTS_MEAN_MODEL_PRODUCTION.pkl')
pts_var_model = joblib.load('../MODELS/SAVED_MODELS/NGBOOST_PTS_VAR_MODEL_PRODUCTION.pkl')
calibration_factor = joblib.load('../MODELS/SAVED_MODELS/NGBOOST_PTS_CALIBRATION_FACTOR_PRODUCTION.pkl')

model = (pts_mean_model, pts_var_model, calibration_factor)  
features = joblib.load('../MODELS/SAVED_MODELS/feature_list.pkl')

print(f"Loaded models with calibration factor: {calibration_factor}")

Loaded models with calibration factor: 4.5


### Load Player Data and Bookmaker Data

In [3]:
pd.set_option('display.max_columns', None)
today = datetime.today().strftime('%Y%m%d')  
current_date = datetime.now().strftime('%Y-%m-%d')

s26 = pd.read_csv('../DATA/CSV_FILES/TRAIN_DATA/PTS_TRAIN_26.csv').sort_values(by='GAME_DATE')

usData = pd.read_csv(f'../DATA/CSV_FILES/PROP_DATA/PLAYER_LINES/NBA_US_{today}.csv')
dfsData = pd.read_csv(f'../DATA/CSV_FILES/PROP_DATA/PLAYER_LINES/NBA_DFS_{today}.csv')

dfsData.head()

/var/folders/9q/5_554qsx5z70w9d_vkmvjg0h0000gn/T/ipykernel_52376/1321450873.py:5: DtypeWarning: Columns (33) have mixed types. Specify dtype option on import or set low_memory=False.
  s26 = pd.read_csv('../DATA/CSV_FILES/TRAIN_DATA/PTS_TRAIN_26.csv').sort_values(by='GAME_DATE')


,BOOKMAKER,CATEGORY,NAME,OVER/UNDER,LINE,ODDS,COMMENCE_TIME,LAST_UPDATE
0,PrizePicks,player_points,Jalen Johnson,Over,23.5,-137,2025-11-23,2025-11-23T21:31:12Z
1,PrizePicks,player_points,Jalen Johnson,Under,23.5,-137,2025-11-23,2025-11-23T21:31:12Z
2,PrizePicks,player_points,Miles Bridges,Over,21.5,-137,2025-11-23,2025-11-23T21:31:12Z
3,PrizePicks,player_points,Miles Bridges,Under,21.5,-137,2025-11-23,2025-11-23T21:31:12Z
4,PrizePicks,player_points,Kon Knueppel,Over,19.5,-137,2025-11-23,2025-11-23T21:31:12Z


### Top EVs for single bets

In [4]:
singlePTSBookies = usData[(usData['CATEGORY'] == 'player_points') & (usData['BOOKMAKER'] != 'Bovada') & (usData['BOOKMAKER'] != 'BetOnline.ag')]

singleBets = calculateSingleBets(s26, singlePTSBookies, model, features, current_date, 
                           edge_threshold=4, stake=10, max_player_appearances=1, top_n=10)



singleBets = singleBets[['NAME', 'BOOKMAKER','LINE', 'PREDICTION', 'SIDE','ODDS','RECOMMENDATION', 'EV%', 'KELLY_FRACTION','SIGMA FLAG']].head(15)
singleBets.to_csv(f'../DATA/CSV_FILES/PROP_DATA/PROPS_EV/singleBets.csv', index=False)
singleBets.head(5)

Processing single bets...
Pre-computing predictions for 95 unique players...
Error getting prediction for LeBron James: float division by zero


,NAME,BOOKMAKER,LINE,PREDICTION,SIDE,ODDS,RECOMMENDATION,EV%,KELLY_FRACTION,SIGMA FLAG
485,Ivica Zubac,BetRivers,15.5,11.88,Under,100,0,49.67,0.497,Med
624,Rui Hachimura,BetRivers,12.5,15.43,Over,120,0,48.47,0.404,High
664,Dillon Brooks,BetRivers,20.5,23.81,Over,114,0,44.91,0.394,High
11,Dyson Daniels,FanDuel,12.5,9.64,Under,-106,0,41.63,0.441,Low
124,Keyonte George,FanDuel,20.5,24.80,Over,-106,1,41.46,0.439,High


## Top EVs for 2 leg bets

### Underdog picks

In [5]:
dfsPTS = dfsData[(dfsData['BOOKMAKER'] == 'Underdog') & (dfsData['CATEGORY'] == 'player_points')]

underdogPairs = calculate2LegBets(s26, dfsPTS, model, features, current_date, 
                           edge_threshold=4, stake=10, max_player_appearances=1, top_n=10)


underdogPairs = underdogPairs[['NAME 1', 'NAME 2', 'LINE 1', 'LINE 2', 'PREDICTION 1', 'PREDICTION 2', 'PROB 1', 'PROB 2', 'MODEL SIDE 1', 'MODEL SIDE 2', 'RECOMMENDATION','EV%', 'KELLY FULL', 'SIGMA FLAG 1', 'SIGMA FLAG 2']]
underdogPairs.to_csv(f'../DATA/CSV_FILES/PROP_DATA/PROPS_EV/underdogPairs.csv', index=False)
underdogPairs

Pre-computing predictions for 54 players...
Error getting prediction for LeBron James: float division by zero
Processing 48 players...
Generated 1054 valid 2-leg combinations


,NAME 1,NAME 2,LINE 1,LINE 2,PREDICTION 1,PREDICTION 2,PROB 1,PROB 2,MODEL SIDE 1,MODEL SIDE 2,RECOMMENDATION,EV%,KELLY FULL,SIGMA FLAG 1,SIGMA FLAG 2
130,Dyson Daniels,Dillon Brooks,12.5,18.5,9.64,23.81,0.729,0.770,under,over,0,64.92,0.325,Low,High
711,James Harden,Rui Hachimura,24.5,11.5,20.91,15.43,0.727,0.728,under,over,0,55.76,0.279,Med,High
957,Keyonte George,Harrison Barnes,20.5,13.5,24.80,16.76,0.728,0.699,over,over,0,49.59,0.248,High,High
850,Deni Avdija,Kevin Love,23.5,4.5,26.88,7.45,0.697,0.726,over,over,0,48.88,0.244,High,Low
890,Donovan Clingan,Jake LaRavia,9.5,7.5,12.47,11.15,0.693,0.719,over,over,0,46.44,0.232,Med,High
422,Brandon Ingram,Shai Gilgeous-Alexander,20.5,31.5,23.75,29.12,0.682,0.650,over,under,0,30.32,0.152,High,High
305,Anthony Black,Ziaire Williams,13.5,10.5,15.81,8.37,0.637,0.638,over,under,0,19.45,0.097,High,High
582,Noah Clowney,Kawhi Leonard,12.5,18.5,14.66,20.38,0.630,0.637,over,over,0,17.89,0.089,High,Med
739,Evan Mobley,Ace Bailey,19.5,11.5,17.57,9.60,0.627,0.626,under,under,0,15.55,0.078,Med,Med
21,Jalen Johnson,Jerami Grant,22.5,18.5,24.50,16.45,0.621,0.619,over,under,0,13.00,0.065,High,High


### Prizepicks picks

In [6]:
dfsPTS = dfsData[(dfsData['BOOKMAKER'] == 'PrizePicks') & (dfsData['CATEGORY'] == 'player_points')]

prizepicksPairs = calculate2LegBets(s26, dfsPTS, model, features, current_date, 
                           edge_threshold=4, stake=10, max_player_appearances=1, top_n=10)


pairsPrizepicks = prizepicksPairs[['NAME 1', 'NAME 2', 'LINE 1', 'LINE 2', 'PREDICTION 1', 'PREDICTION 2', 'PROB 1', 'PROB 2', 'MODEL SIDE 1', 'MODEL SIDE 2', 'RECOMMENDATION','EV%', 'KELLY FULL', 'SIGMA FLAG 1', 'SIGMA FLAG 2']].head(10)
pairsPrizepicks.to_csv(f'../DATA/CSV_FILES/PROP_DATA/PROPS_EV/prizepicksPairs.csv', index=False)
prizepicksPairs

Pre-computing predictions for 90 players...
Error getting prediction for LeBron James: float division by zero
Processing 83 players...
Generated 3193 valid 2-leg combinations


,NAME 1,NAME 2,LINE 1,LINE 2,PREDICTION 1,PREDICTION 2,MODEL SIDE 1,MODEL SIDE 2,PROB 1,PROB 2,PROB BOTH,EDGE 1,EDGE 2,COMBINED EDGE,EV%,KELLY FULL,RECOMMENDATION,SIGMA 1,SIGMA 2,SIGMA FLAG 1,SIGMA FLAG 2,CI 1,CI 2,CORRELATION,SAME_GAME,EXPECTED ROI
2307,Ivica Zubac,Dillon Brooks,15.5,18.5,11.88,23.81,under,over,0.748,0.770,0.5645,0.170,0.192,0.242,69.35,0.347,0,5.42,7.19,Med,High,"(1.3, 22.5)","(9.7, 37.9)",0.05,0,69.3
583,Dyson Daniels,Austin Reaves,12.5,22.5,9.64,26.89,under,over,0.729,0.730,0.5212,0.151,0.152,0.198,56.36,0.282,0,4.70,7.18,Low,High,"(0.4, 18.8)","(12.8, 41.0)",0.05,0,56.4
2080,James Harden,Rui Hachimura,24.5,11.5,20.91,15.43,under,over,0.727,0.728,0.5192,0.149,0.150,0.196,55.76,0.279,0,5.94,6.46,Med,High,"(9.3, 32.6)","(2.8, 28.1)",0.05,0,55.8
2971,Keyonte George,Harrison Barnes,20.5,13.5,24.80,16.76,over,over,0.728,0.699,0.4986,0.150,0.121,0.175,49.59,0.248,0,7.09,6.25,High,High,"(10.9, 38.7)","(4.5, 29.0)",0.05,0,49.6
2609,Deni Avdija,Jake LaRavia,23.5,7.5,26.88,11.15,over,over,0.697,0.719,0.4915,0.119,0.141,0.167,47.46,0.237,0,6.55,6.29,High,High,"(14.1, 39.7)","(0.0, 23.5)",0.05,0,47.5
2822,Donovan Clingan,Kevin Love,9.5,5.0,12.47,7.45,over,over,0.693,0.691,0.4691,0.114,0.113,0.145,40.73,0.204,0,5.91,4.91,Med,Low,"(0.9, 24.1)","(0.0, 17.1)",0.05,0,40.7
1513,Brandon Ingram,Devin Vassell,20.5,17.5,23.75,14.77,over,under,0.682,0.661,0.4421,0.104,0.083,0.117,32.62,0.163,0,6.86,6.57,High,High,"(10.3, 37.2)","(1.9, 27.7)",0.05,0,32.6
643,Ryan Kalkbrenner,Shai Gilgeous-Alexander,8.5,31.5,7.11,29.12,under,under,0.654,0.650,0.4165,0.076,0.071,0.091,24.95,0.125,0,3.50,6.21,Low,High,"(0.3, 14.0)","(17.0, 41.3)",0.05,0,24.9
1389,Goga Bitadze,Marcus Smart,6.5,6.5,8.27,8.76,over,over,0.649,0.647,0.4116,0.071,0.069,0.086,23.49,0.117,0,4.64,5.97,Low,Med,"(0.0, 17.4)","(0.0, 20.5)",0.05,0,23.5
1697,Ziaire Williams,Kawhi Leonard,10.5,18.5,8.37,20.38,under,over,0.638,0.637,0.3982,0.060,0.059,0.072,19.45,0.097,0,6.04,5.37,High,Med,"(0.0, 20.2)","(9.8, 30.9)",0.05,0,19.4


## 3 leg parlay

### Underdog picks

In [7]:
dfsPTS = dfsData[(dfsData['BOOKMAKER'] == 'Underdog') & (dfsData['CATEGORY'] == 'player_points')]

underdogTrios = calculate3LegBets(s26, dfsPTS, model, features, current_date, 
                           edge_threshold=4, stake=10, max_player_appearances=1, top_n=10)

underdogTrios = underdogTrios[['NAME 1', 'NAME 2', 'NAME 3', 'LINE 1', 'LINE 2', 'LINE 3', 'PREDICTION 1', 'PREDICTION 2', 'PREDICTION 3', 'PROB 1', 'PROB 2', 'PROB 3', 'MODEL SIDE 1', 'MODEL SIDE 2', 'MODEL SIDE 3', 'RECOMMENDATION','EV%', 'KELLY FULL', 'SIGMA FLAG 1', 'SIGMA FLAG 2', 'SIGMA FLAG 3']].head(10)
underdogTrios.to_csv(f'../DATA/CSV_FILES/PROP_DATA/PROPS_EV/underdogTrios.csv', index=False)
underdogTrios.head()

Pre-computing predictions for 54 players...
Error getting prediction for LeBron James: float division by zero
Processing 48 players...
Generated 16988 valid 3-leg combinations


,NAME 1,NAME 2,NAME 3,LINE 1,LINE 2,LINE 3,PREDICTION 1,PREDICTION 2,PREDICTION 3,PROB 1,PROB 2,PROB 3,MODEL SIDE 1,MODEL SIDE 2,MODEL SIDE 3,RECOMMENDATION,EV%,KELLY FULL,SIGMA FLAG 1,SIGMA FLAG 2,SIGMA FLAG 3
3009,Dyson Daniels,Rui Hachimura,Dillon Brooks,12.5,11.5,18.5,9.64,15.43,23.81,0.729,0.728,0.770,under,over,over,0,120.65,0.241,Low,High,High
13811,James Harden,Keyonte George,Kevin Love,24.5,20.5,4.5,20.91,24.80,7.45,0.727,0.728,0.726,under,over,over,0,107.58,0.215,Med,High,Low
15567,Deni Avdija,Jake LaRavia,Harrison Barnes,23.5,7.5,13.5,26.88,11.15,16.76,0.697,0.719,0.699,over,over,over,0,89.33,0.179,High,High,High
6708,Anthony Black,Brandon Ingram,Donovan Clingan,13.5,20.5,9.5,15.81,23.75,12.47,0.637,0.682,0.693,over,over,over,0,62.53,0.125,High,High,Med
11293,Ziaire Williams,Kawhi Leonard,Shai Gilgeous-Alexander,10.5,18.5,31.5,8.37,20.38,29.12,0.638,0.637,0.650,under,over,under,0,42.50,0.085,High,Med,High


### Prizepicks picks

In [8]:
dfsPTS = dfsData[(dfsData['BOOKMAKER'] == 'PrizePicks') & (dfsData['CATEGORY'] == 'player_points')]

triosPrizepicks = calculate3LegBets(s26, dfsPTS, model, features, current_date, 
                           edge_threshold=4, stake=10, max_player_appearances=1, top_n=15)


triosPrizepicks = triosPrizepicks[['NAME 1', 'NAME 2', 'NAME 3', 'LINE 1', 'LINE 2', 'LINE 3', 'PREDICTION 1', 'PREDICTION 2', 'PREDICTION 3', 'PROB 1', 'PROB 2', 'PROB 3', 'MODEL SIDE 1', 'MODEL SIDE 2', 'MODEL SIDE 3', 'RECOMMENDATION','EV%', 'KELLY FULL', 'SIGMA FLAG 1', 'SIGMA FLAG 2', 'SIGMA FLAG 3']].head(10)
triosPrizepicks.to_csv(f'../DATA/CSV_FILES/PROP_DATA/PROPS_EV/prizepicksTrios.csv', index=False)
triosPrizepicks.head()

Pre-computing predictions for 90 players...
Error getting prediction for LeBron James: float division by zero
Processing 83 players...
Generated 90320 valid 3-leg combinations


,NAME 1,NAME 2,NAME 3,LINE 1,LINE 2,LINE 3,PREDICTION 1,PREDICTION 2,PREDICTION 3,PROB 1,PROB 2,PROB 3,MODEL SIDE 1,MODEL SIDE 2,MODEL SIDE 3,RECOMMENDATION,EV%,KELLY FULL,SIGMA FLAG 1,SIGMA FLAG 2,SIGMA FLAG 3
76708,Ivica Zubac,Austin Reaves,Dillon Brooks,15.5,22.5,18.5,11.88,26.89,23.81,0.748,0.730,0.770,under,over,over,0,126.98,0.254,Med,High,High
22989,Dyson Daniels,James Harden,Rui Hachimura,12.5,24.5,11.5,9.64,20.91,15.43,0.729,0.727,0.728,under,under,over,0,108.50,0.217,Low,Med,High
83035,Deni Avdija,Keyonte George,Harrison Barnes,23.5,20.5,13.5,26.88,24.80,16.76,0.697,0.728,0.699,over,over,over,0,91.61,0.183,High,High,High
55118,Brandon Ingram,Donovan Clingan,Jake LaRavia,20.5,9.5,7.5,23.75,12.47,11.15,0.682,0.693,0.719,over,over,over,0,83.55,0.167,High,Med,High
26835,Ryan Kalkbrenner,Kevin Love,Devin Vassell,8.5,5.0,17.5,7.11,7.45,14.77,0.654,0.691,0.661,under,over,under,0,61.43,0.123,Low,Low,High


In [ ]:
# df = playerScoring('Trey Murphy III', s26, current_date, teamStarPlayer, projectedStartingFive)
# playerContext('Trey Murphy III', s26, current_date, projectedStartingFive, mainStartingFive, teamStarPlayer)